# Lab 02 model training

In this coding lab, you will load the xBD dataset using a Data Loader and train a Siamese Unet.

# Step 0: Installation

Make sure to use a GPU and have access to internet connection in the Kaggle notebook:

1. On the arrow on the bottom right, select "Notebook Options" and then "Accelerator" to choose the GPU P100, and select "Variables & Files" under Persistence. **Note that Kaggle allows 30h per week per user of accelerated computing. Plan your work accordingly. It takes time to load the data and you may experience unavailability of GPUs or Session Errors**
1. Make sure your Kaggle account is phone verified by clicking "Get phone verified" in the left sidebar under "Notebook options" and following the steps (this step is required to switch on the internet connection needed to install packages). 
1. After phone verification, the full settings menu should be visible. Toggle the "Internet" switch.

More visualizations of the process to connect the notebook to the internet are provided [here](https://stackoverflow.com/questions/68142524/cannot-access-internet-on-kaggle-notebook)


## Requirements:

1. Downloading the tiles of the xBD dataset here: [xbd-tiles-256.zip](https://www.dropbox.com/scl/fi/7g4q0fj73crptgfb37b6c/xbd-tiles-256.zip?rlkey=ikf973l5r51bu554ya94lqtbm&st=hktideas&dl=0). The xBD dataset was tiled even more to allow a small model to learn fast. 
1. Go to the "Side Bar", Click on "Input"
1. Upload as `xbd-tiles-256` the file: `xbd-tiles-256.zip`.
1. Downloading the [utilsxbd.zip](https://www.dropbox.com/scl/fi/65poregqxewhtvdtykyv7/utilsxbd.zip?rlkey=3yzdp5qarubfvxma4eutdy0cj&st=6o73fue3&dl=0) and upload them as `utilsxbd`





In [ ]:
#Check if you have access to a GPU
!nvidia-smi

In [ ]:
# Install Rust
!curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y
# Add to path
import os
os.environ["PATH"] = f"{os.environ['HOME']}/.cargo/bin:{os.environ['PATH']}"
!pip install safetensors
!pip install torch torchvision torchaudio 
!pip install segmentation-models-pytorch
!pip install torchsummary
os.environ['CUDA_LAUNCH_BLOCKING'] = '1'

In [ ]:
import os
import glob
import imageio
import numpy as np
import random
from PIL import Image
import gc
import torch
import torchvision
import torch.nn as nn
import torch.nn.functional as F
from torchsummary import summary
from torch.utils.data import DataLoader
from torch.utils.data import Dataset as BaseDataset
from typing import List
from pathlib import Path
import numpy as np  # For numerical operations and array manipulations
import matplotlib.pyplot as plt  # For creating plots and visualizations
from matplotlib.colors import ListedColormap

Let's check if pytorch can use the available GPU:

In [ ]:
if torch.cuda.is_available():
    device = torch.device("cuda")
    print("PyTorch is using the GPU")
else:
    device = torch.device("cpu")
    print("PyTorch is using the CPU")

## Step 0 - Overview of the data

Splitting the dataset by pre-, post-image, building mask and damage assessment mask. 

In [ ]:
# Define paths
dataset_path = '/kaggle/input/xbd-tiles-256/xbd-tiles-256'
train_images_path = os.path.join(dataset_path, 'train', 'images')

# Get filenames only once
all_image_files = os.listdir(train_images_path)

# Extract unique base filenames
filenames = set()
for filename in all_image_files:
    if '_pre' in filename:
        base_name = filename.split('_pre')[0]
    elif '_post' in filename:
        base_name = filename.split('_post')[0]
    filenames.add(base_name)

# Convert to list for indexing
filenames = list(filenames)

# Create paths for all images and masks
pre_image_list = [os.path.join(dataset_path, 'train', "images", f"{filename}_pre_disaster.tif") for filename in filenames]
post_image_list = [os.path.join(dataset_path, 'train', "images", f"{filename}_post_disaster.tif") for filename in filenames]
building_mask_list = [os.path.join(dataset_path, 'train', "masks", f"{filename}_pre_disaster.png") for filename in filenames]
damage_mask_list = [os.path.join(dataset_path, 'train', "masks", f"{filename}_post_disaster.png") for filename in filenames]

In [ ]:
def visualize_disaster_assessment(pre_img, post_img, building_label, damage_label, normalize=True):
    """
    Visualize a disaster assessment with pre/post disaster images, building footprint, and damage assessment.

    Args:
        pre_img (numpy.ndarray): Pre-disaster image (H, W, C), pixel values in [0, 255].
        post_img (numpy.ndarray): Post-disaster image (H, W, C), pixel values in [0, 255].
        building_label (numpy.ndarray): Binary building footprint mask (0=background, 1=building).
        damage_label (numpy.ndarray): Damage assessment mask with values 0–4:
                                      0=no building, 1=no damage, 2=minor, 3=major, 4=destroyed.
        normalize (bool): If True, divides image pixel values by 255 to map them to [0, 1]
                          as required by matplotlib's imshow. Set to False if values are
                          already in [0, 1].
    """
    fig, axes = plt.subplots(1, 4, figsize=(20, 5))

    # Helper: convert a raw image to a display-safe [0, 1] float array.
    # np.clip guards against any float rounding that pushes values slightly out of range,
    # which would otherwise cause matplotlib warnings.
    def to_display(img):
        return np.clip(img / 255.0, 0, 1) if normalize else img

    # 1. Pre-disaster image
    # Images loaded with imageio are already in (H, W, C) format — no transpose needed
    axes[0].imshow(to_display(pre_img))
    axes[0].set_title("Pre-disaster Image", fontsize=14)
    axes[0].axis('off')

    # 2. Post-disaster image
    axes[1].imshow(to_display(post_img))
    axes[1].set_title("Post-disaster Image", fontsize=14)
    axes[1].axis('off')

    # 3. Building footprint mask (binary: 0=background, 1=building)
    # Use a simple two-colour map so the meaning is unambiguous
    building_cmap = ListedColormap(['black', 'white'])
    bld_im = axes[2].imshow(building_label, cmap=building_cmap, vmin=0, vmax=1)
    axes[2].set_title("Building Footprint", fontsize=14)
    axes[2].axis('off')
    bcbar = plt.colorbar(bld_im, ax=axes[2], ticks=[0, 1], fraction=0.046, pad=0.04)
    bcbar.set_ticklabels(['Background', 'Building'])
    bcbar.set_label('Building', rotation=270, labelpad=15)

    # 4. Damage assessment mask (0–4 categorical scale)
    # Use a fixed colour per class so the legend is stable across different samples
    damage_colors = ['black', 'green', 'yellow', 'orange', 'red']
    damage_cmap = ListedColormap(damage_colors)
    # Cast to uint8 — values are 0–4, no need for int32
    dam_im = axes[3].imshow(damage_label.astype(np.uint8), cmap=damage_cmap, vmin=0, vmax=4)
    axes[3].set_title("Damage Assessment", fontsize=14)
    axes[3].axis('off')
    dcbar = plt.colorbar(dam_im, ax=axes[3], ticks=[0, 1, 2, 3, 4], fraction=0.046, pad=0.04)
    dcbar.set_ticklabels(['No building', 'No damage', 'Minor', 'Major', 'Destroyed'])
    dcbar.set_label('Damage Level', rotation=270, labelpad=15)

    plt.tight_layout()
    plt.show()

In [ ]:
def normalize_img(
    img: np.ndarray, 
    mean: List[float] = [123.675, 116.28, 103.53], 
    std: List[float] = [58.395, 57.12, 57.375]
) -> np.ndarray:
    """Normalize image with mean and standard deviation.

    This function normalizes an image by subtracting the mean and dividing by 
    the standard deviation for each color channel. The normalization follows
    the formula: (pixel - mean) / std.
    
    Args:
        img (np.array): Image to normalize.
        mean (list): Mean values.
        std (list): Standard deviation values.
    Returns:
        proc_img (np.array): Normalized image
    """
    imgarr = np.asarray(img)
    proc_img = np.empty_like(imgarr, np.float32)

    proc_img[..., 0] = (imgarr[..., 0] - mean[0]) / std[0]
    proc_img[..., 1] = (imgarr[..., 1] - mean[1]) / std[1]
    proc_img[..., 2] = (imgarr[..., 2] - mean[2]) / std[2]
    return proc_img

# Step 1: Create the Data Class

1. Develop a custom xBD Pytorch Dataset that loads images and performs augmentation, preprocessing and normalises the dataset using the Imagenet statics. Follow the [Pytorch Documentation](https://pytorch.org/tutorials/beginner/basics/data_tutorial.html#creating-a-custom-dataset-for-your-files) for the structure of the Custom Dataset.

In [ ]:
class xBD_Dataset(BaseDataset):
    """xBD Dataset: reads images, applies augmentation and preprocessing."""

    def __init__(self, dataset_path, split='train', augmentation=None, preprocessing=None):
        # 'split' indicates which subset to load: 'train', 'val', or 'test'
        self.split = split
        # Optional augmentation function applied to images and labels (e.g. random flips)
        self.augmentation = augmentation
        # Optional preprocessing function applied after augmentation (e.g. custom transforms)
        self.preprocessing = preprocessing

        # Build paths to the images and masks directories for this split
        img_dir = Path(dataset_path) / split / 'images'
        msk_dir = Path(dataset_path) / split / 'masks'

        # Each file is named like "event_id_pre_disaster.tif" or "event_id_post_disaster.tif".
        # We extract the unique base name (e.g. "event_id") by stripping the known suffixes.
        # Path(f).stem removes the file extension, then we strip the disaster suffix.
        # sorted() ensures a consistent, reproducible ordering across runs (set() is unordered).
        filenames = sorted(set(
            Path(f).stem.replace('_pre_disaster', '').replace('_post_disaster', '')
            for f in os.listdir(img_dir)
        ))

        # Store all four paths for each sample as a list of tuples.
        # Using tuples keeps pre/post images and their masks guaranteed to stay aligned by index,
        # unlike four separate lists which could silently get out of sync.
        self.samples = [
            (img_dir / f"{f}_pre_disaster.tif",   # pre-disaster image
             img_dir / f"{f}_post_disaster.tif",   # post-disaster image
             msk_dir / f"{f}_pre_disaster.png",    # building footprint mask
             msk_dir / f"{f}_post_disaster.png")   # damage assessment mask
            for f in filenames
        ]

    def __getitem__(self, index):
        # Unpack the four paths for this sample from the tuple
        pre_path, post_path, bld_path, dam_path = self.samples[index]

        # Load images as float32 — required for normalization arithmetic below
        pre_img  = imageio.v3.imread(pre_path).astype(np.float32)
        post_img = imageio.v3.imread(post_path).astype(np.float32)

        # Load masks as uint8 — labels are integer class indices (0–4), not continuous values,
        # so float32 would waste memory and be semantically incorrect
        building_label = imageio.v3.imread(bld_path).astype(np.uint8)
        damage_label   = imageio.v3.imread(dam_path).astype(np.uint8)

        # Apply augmentation if provided (e.g. random flips, rotations).
        # We check for the function rather than the split name so the caller controls
        # whether augmentation is active — simply don't pass it for val/test sets.
        if self.augmentation:
            pre_img, post_img, building_label, damage_label = self.augmentation(
                pre_img, post_img, building_label, damage_label)

        # Apply any additional preprocessing if provided
        if self.preprocessing:
            pre_img, post_img, building_label, damage_label = self.preprocessing(
                pre_img, post_img, building_label, damage_label)

        # Normalize images using ImageNet mean and std — this matches the statistics
        # the ResNet encoder was pretrained on, helping transfer learning work better
        pre_img  = normalize_img(pre_img)
        post_img = normalize_img(post_img)

        # PyTorch conv layers expect (channel, height, width) but images are loaded as
        # (height, width, channel) — np.moveaxis moves the last axis (-1) to position 0
        pre_img  = np.moveaxis(pre_img,  -1, 0)  # (H, W, C) → (C, H, W)
        post_img = np.moveaxis(post_img, -1, 0)  # (H, W, C) → (C, H, W)

        return pre_img, post_img, building_label, damage_label

    def __len__(self):
        # Returns the total number of samples in this split
        return len(self.samples)

The class `transform` contains methods for various image transformations. These transformations are often used in data augmentation for training machine learning models, particularly in the field of computer vision. Here's a brief overview of the methods:

* normalize_img: Normalizes an image by subtracting the mean and dividing by the standard deviation for each color channel.

* _img_rescaling: Rescales an image and its corresponding labels (location and damage) to a new scale.

* random_scaling: Randomly scales an image and its labels within a given range.

* img_resize_short: Resizes the shorter edge of an image to a minimum size.

* random_resize: Randomly resizes an image and its label within a given size range.

* random_fliplr and random_flipud: Half of the time flips an image and its labels horizontally (left-right) or vertically (up-down).

* random_rot: Randomly rotates an image and its labels by a multiple of 90 degrees.

* random_crop: Randomly crops an image and its labels to a specified size.

* colormap: Generates a colormap with N colors.

* encode_cmap: Encodes a label image using a colormap.

* tensorboard_image: Prepares images, predictions, and labels for display in TensorBoard, a visualization tool for machine learning. It normalizes the input images, encodes the predictions and labels with a colormap, and arranges them in a grid for display.

2. Implement the `random_fliplr`, `random_flipud`, and `random_rot` using `np.fliplr`, `np.flipud` and `np.rot90`

In [ ]:
class transform(object):

    def _img_rescaling(self, pre_img, post_img, loc_label, dam_label, scale=None):
        # Guard: if no labels are provided, only resize the images
        # This check must come before accessing dam_label.shape below
        if dam_label is None:
            new_pre_img = Image.fromarray(pre_img.astype(np.uint8)).resize([int(scale * pre_img.shape[1]), int(scale * pre_img.shape[0])], resample=Image.BILINEAR)
            new_post_img = Image.fromarray(post_img.astype(np.uint8)).resize([int(scale * post_img.shape[1]), int(scale * post_img.shape[0])], resample=Image.BILINEAR)
            return np.asarray(new_pre_img).astype(np.float32), np.asarray(new_post_img).astype(np.float32)

        h, w = dam_label.shape
        new_scale = [int(scale * w), int(scale * h)]

        # Resize images with bilinear interpolation — smooth for continuous pixel values
        new_pre_img = Image.fromarray(pre_img.astype(np.uint8)).resize(new_scale, resample=Image.BILINEAR)
        new_pre_img = np.asarray(new_pre_img).astype(np.float32)

        new_post_img = Image.fromarray(post_img.astype(np.uint8)).resize(new_scale, resample=Image.BILINEAR)
        new_post_img = np.asarray(new_post_img).astype(np.float32)

        # Resize masks with nearest-neighbour — preserves integer class indices (no interpolation)
        new_dam_label = np.asarray(Image.fromarray(dam_label).resize(new_scale, resample=Image.NEAREST))
        new_loc_label = np.asarray(Image.fromarray(loc_label).resize(new_scale, resample=Image.NEAREST))

        return new_pre_img, new_post_img, new_loc_label, new_dam_label


    def random_scaling(self, pre_img, post_img, loc_label, dam_label, size_range, scale_range):
        min_ratio, max_ratio = scale_range
        assert min_ratio <= max_ratio

        # Sample a random scale ratio within the given range
        ratio = random.uniform(min_ratio, max_ratio)

        return self._img_rescaling(pre_img, post_img, loc_label, dam_label, scale=ratio)


    def img_resize_short(self, image, min_size=512):
        h, w, _ = image.shape
        # Only upscale if the shorter side is smaller than min_size
        if min(h, w) >= min_size:
            return image

        scale = float(min_size) / min(h, w)
        new_scale = [int(scale * w), int(scale * h)]

        new_image = Image.fromarray(image.astype(np.uint8)).resize(new_scale, resample=Image.BILINEAR)
        return np.asarray(new_image).astype(np.float32)


    def random_crop(self, pre_img, post_img, loc_label, dam_label, crop_size, mean_rgb=None, ignore_index=255):
        # Avoid mutable default argument — a list default is shared across all calls in Python
        if mean_rgb is None:
            mean_rgb = [0, 0, 0]

        h, w = loc_label.shape

        # If the image is smaller than crop_size, pad it first
        H = max(crop_size, h)
        W = max(crop_size, w)

        # Initialise padded arrays filled with the mean RGB value (neutral padding)
        pad_pre_image  = np.full((H, W, 3), mean_rgb, dtype=np.float32)
        pad_post_image = np.full((H, W, 3), mean_rgb, dtype=np.float32)
        # Initialise label padding with ignore_index so padded pixels are excluded from the loss
        pad_loc_label = np.full((H, W), ignore_index, dtype=np.float32)
        pad_dam_label = np.full((H, W), ignore_index, dtype=np.float32)

        # Place the original image/mask at a random position inside the padded canvas
        H_pad = int(np.random.randint(H - h + 1))
        W_pad = int(np.random.randint(W - w + 1))

        pad_pre_image[H_pad:(H_pad + h),  W_pad:(W_pad + w), :] = pre_img
        pad_post_image[H_pad:(H_pad + h), W_pad:(W_pad + w), :] = post_img
        pad_loc_label[H_pad:(H_pad + h),  W_pad:(W_pad + w)]    = loc_label
        pad_dam_label[H_pad:(H_pad + h),  W_pad:(W_pad + w)]    = dam_label

        def get_random_cropbox(cat_max_ratio=0.75):
            # Try up to 10 times to find a crop where no single class dominates more
            # than cat_max_ratio of the pixels — avoids crops with no useful signal
            for i in range(10):
                H_start = random.randrange(0, H - crop_size + 1, 1)
                H_end   = H_start + crop_size
                W_start = random.randrange(0, W - crop_size + 1, 1)
                W_end   = W_start + crop_size

                temp_label = pad_loc_label[H_start:H_end, W_start:W_end]
                index, cnt = np.unique(temp_label, return_counts=True)
                cnt = cnt[index != ignore_index]
                if len(cnt > 1) and np.max(cnt) / np.sum(cnt) < cat_max_ratio:
                    break

            return H_start, H_end, W_start, W_end

        H_start, H_end, W_start, W_end = get_random_cropbox()

        pre_img   = pad_pre_image[H_start:H_end,  W_start:W_end, :]
        post_img  = pad_post_image[H_start:H_end, W_start:W_end, :]
        loc_label = pad_loc_label[H_start:H_end,  W_start:W_end]
        dam_label = pad_dam_label[H_start:H_end,  W_start:W_end]

        return pre_img, post_img, loc_label, dam_label


    def colormap(self, N=256, normalized=False):
        # Generates a colour palette using the Pascal VOC bit-interleaving scheme
        def bitget(byteval, idx):
            return ((byteval & (1 << idx)) != 0)

        dtype = 'float32' if normalized else 'uint8'
        cmap = np.zeros((N, 3), dtype=dtype)
        for i in range(N):
            r = g = b = 0
            c = i
            for j in range(8):
                r = r | (bitget(c, 0) << 7 - j)
                g = g | (bitget(c, 1) << 7 - j)
                b = b | (bitget(c, 2) << 7 - j)
                c = c >> 3
            cmap[i] = np.array([r, g, b])

        cmap = cmap / 255 if normalized else cmap
        return cmap


    def encode_cmap(self, label):
        # Maps integer class indices to RGB colours for visualisation
        cmap = self.colormap()
        return cmap[label.astype(np.int16), :]


    def tensorboard_image(self, inputs=None, outputs=None, labels=None, bgr=None):
        # De-normalise inputs by adding back the BGR mean before display
        inputs[:, 0, :, :] = inputs[:, 0, :, :] + bgr[0]
        inputs[:, 1, :, :] = inputs[:, 1, :, :] + bgr[1]
        inputs[:, 2, :, :] = inputs[:, 2, :, :] + bgr[2]
        inputs = inputs[:, [2, 1, 0], :, :].type(torch.uint8)
        grid_inputs = torchvision.utils.make_grid(tensor=inputs, nrow=2)

        # Convert predicted logits to class indices, then map to colours
        preds = torch.argmax(outputs, dim=1).cpu().numpy()
        preds_cmap = self.encode_cmap(preds)
        # .contiguous() ensures consecutive memory layout — required for DataParallel/DDP
        preds_cmap = torch.from_numpy(preds_cmap).permute([0, 3, 1, 2]).contiguous()
        grid_outputs = torchvision.utils.make_grid(tensor=preds_cmap, nrow=2)

        labels_cmap = self.encode_cmap(labels.cpu().numpy())
        labels_cmap = torch.from_numpy(labels_cmap).permute([0, 3, 1, 2]).contiguous()
        grid_labels = torchvision.utils.make_grid(tensor=labels_cmap, nrow=2)

        return grid_inputs, grid_outputs, grid_labels


    def random_fliplr(self, pre_img, post_img, loc_label, dam_label):
        # Randomly flip all arrays horizontally (left ↔ right) with 50% probability.
        # np.fliplr returns a view with a negative stride — PyTorch cannot wrap such arrays
        # into tensors, so we call .copy() to return a contiguous array.
        if random.random() > 0.5:
            pre_img   = np.fliplr(pre_img).copy()
            post_img  = np.fliplr(post_img).copy()
            loc_label = np.fliplr(loc_label).copy()
            dam_label = np.fliplr(dam_label).copy()

        return pre_img, post_img, loc_label, dam_label


    def random_flipud(self, pre_img, post_img, loc_label, dam_label):
        # Randomly flip all arrays vertically (up ↔ down) with 50% probability.
        # Same .copy() requirement as random_fliplr above.
        if random.random() > 0.5:
            pre_img   = np.flipud(pre_img).copy()
            post_img  = np.flipud(post_img).copy()
            loc_label = np.flipud(loc_label).copy()
            dam_label = np.flipud(dam_label).copy()

        return pre_img, post_img, loc_label, dam_label


    def random_rot(self, pre_img, post_img, loc_label, dam_label):
        # Randomly rotate all arrays by 90, 180, or 270 degrees (k ∈ {1, 2, 3}).
        # np.rot90 also returns a view, so .copy() is needed here too.
        k = random.randrange(3) + 1  # k=1 → 90°, k=2 → 180°, k=3 → 270°
        pre_img   = np.rot90(pre_img,   k).copy()
        post_img  = np.rot90(post_img,  k).copy()
        loc_label = np.rot90(loc_label, k).copy()
        dam_label = np.rot90(dam_label, k).copy()

        return pre_img, post_img, loc_label, dam_label

In [ ]:
# Instantiate transform once — no need to create a new object on every call
_t = transform()

def train_transform(pre_img, post_img, building_label, damage_label):
    # Apply a random horizontal flip (left ↔ right) with 50% probability
    pre_img, post_img, building_label, damage_label = _t.random_fliplr(
        pre_img=pre_img, post_img=post_img, loc_label=building_label, dam_label=damage_label)

    # Apply a random vertical flip (up ↔ down) with 50% probability
    pre_img, post_img, building_label, damage_label = _t.random_flipud(
        pre_img=pre_img, post_img=post_img, loc_label=building_label, dam_label=damage_label)

    # Apply a random 90°/180°/270° rotation
    pre_img, post_img, building_label, damage_label = _t.random_rot(
        pre_img=pre_img, post_img=post_img, loc_label=building_label, dam_label=damage_label)

    return pre_img, post_img, building_label, damage_label

In [ ]:
# Reload a fresh (un-augmented) copy of the sample so that re-running this cell
# always starts from the original image rather than stacking transforms on top of
# a previously transformed version.
index = 0
pre_img_orig  = imageio.v3.imread(pre_image_list[index]).astype(np.float32)
post_img_orig = imageio.v3.imread(post_image_list[index]).astype(np.float32)
bld_label_orig = imageio.v3.imread(building_mask_list[index]).astype(np.float32)
dam_label_orig = imageio.v3.imread(damage_mask_list[index]).astype(np.float32)

# Apply the training augmentation pipeline (random flip lr, flip ud, rotation)
pre_aug, post_aug, bld_aug, dam_aug = train_transform(
    pre_img_orig, post_img_orig, bld_label_orig, dam_label_orig)

# Visualise the result — compare with the original visualisation in the cell above.
# You should see the image randomly flipped and/or rotated.
# Run this cell multiple times to observe different augmentation outcomes.
visualize_disaster_assessment(pre_aug, post_aug, bld_aug, dam_aug, normalize=True)

# Step 2: Load the data

In [ ]:
# Root path to the xBD dataset — this must match where you uploaded xbd-tiles-256.zip
# on Kaggle: sidebar → Input → your dataset name → copy the path shown there
root_path = Path('/kaggle/input/xbd-tiles-256/xbd-tiles-256')

# Validate early: if the path is wrong, fail here with a clear message rather than
# getting a cryptic error later when the DataLoader tries to read files
assert root_path.exists(), (
    f"Dataset not found at '{root_path}'. "
    "Make sure you have uploaded xbd-tiles-256.zip as an input dataset on Kaggle."
)

3. Use the `xBD_Dataset` to load `train_set` and `val_set`.

In [ ]:
# Training set — augmentation is applied to every sample during training
# to artificially increase data diversity and reduce overfitting
train_set = xBD_Dataset(root_path, split='train', augmentation=train_transform)

# Validation set — no augmentation, we want to evaluate on unmodified images
# to get a reliable estimate of real-world performance
val_set = xBD_Dataset(root_path, split='val')

4. Use the [torch.DataLoader](https://pytorch.org/docs/stable/data.html#torch.utils.data.DataLoader) setting the paramenter: `batch_size`, `shuffle`, `num_workers` and `pin_memory`. 

In [ ]:
def collate_fn(batch):
    """
    Custom collate function for the DataLoader.

    Why do we need a custom one?
    The default PyTorch collate_fn would leave labels as torch.uint8 tensors (matching
    the numpy dtype we load them as). However, nn.CrossEntropyLoss requires class index
    targets to be torch.long (int64). We handle that conversion here, once per batch,
    rather than scattering .long() calls throughout the training loop.

    The .copy() calls ensure each array owns its memory (no negative-stride views),
    which is required before wrapping a numpy array in a torch.Tensor.
    """
    # Unzip the list of (pre, post, building, damage) tuples into four separate tuples
    pre_imgs, post_imgs, building_labels, damage_labels = zip(*batch)

    # Stack images into a single (B, C, H, W) float tensor
    pre_imgs        = torch.stack([torch.from_numpy(img.copy()).float()  for img in pre_imgs])
    post_imgs       = torch.stack([torch.from_numpy(img.copy()).float()  for img in post_imgs])

    # Stack labels into a single (B, H, W) long tensor — CrossEntropyLoss requires int64
    building_labels = torch.stack([torch.from_numpy(lbl.copy()).long()   for lbl in building_labels])
    damage_labels   = torch.stack([torch.from_numpy(lbl.copy()).long()   for lbl in damage_labels])

    return pre_imgs, post_imgs, building_labels, damage_labels

In [ ]:
# Training DataLoader
# - shuffle=True: randomises sample order each epoch, reducing the risk of the model
#   learning spurious patterns from a fixed sequence
# - num_workers=0 
# - pin_memory=True: pre-allocates batches in page-locked (pinned) CPU memory,
#   which speeds up the CPU→GPU transfer
train_loader = DataLoader(
    train_set,
    batch_size=32,
    shuffle=True,
    num_workers=0,
    pin_memory=True,
    collate_fn=collate_fn,
)

# Validation DataLoader
# - shuffle=False: keeps a consistent sample order so metrics are reproducible
#   and comparable across epochs
val_loader = DataLoader(
    val_set,
    batch_size=32,
    shuffle=False,
    num_workers=0,
    pin_memory=True,
    collate_fn=collate_fn,
)

In [ ]:
# Find the first sample that contains damaged buildings (damage class > 1).
# We use next() with a generator so we stop scanning as soon as one is found —
# no need to read every mask in the dataset.
# damage classes: 0=no building, 1=no damage, 2=minor, 3=major, 4=destroyed
first_damaged_index = next(
    i for i, (_, _, _, dam_path) in enumerate(train_set.samples)
    if np.any(imageio.v3.imread(dam_path) > 1)
)
print(f"First sample with damaged buildings: index {first_damaged_index}")

In [ ]:
# Sanity-check: visualise a processed sample directly from the dataset object.
# This confirms that loading, augmentation, and normalization are all working correctly.
index = 32
pre_img, post_img, building_label, damage_label = train_set[index]

# __getitem__ returns images in (C, H, W) as required by PyTorch conv layers.
# imshow expects (H, W, C), so we move the channel axis back to the last position.
pre_img  = np.moveaxis(pre_img,  0, -1)  # (C, H, W) → (H, W, C)
post_img = np.moveaxis(post_img, 0, -1)

# normalize_img() inside __getitem__ shifted pixel values to roughly [-2, 2].
# Matplotlib clips floats outside [0, 1], turning most pixels black.
# We reverse the normalization (denormalize) to recover the original [0, 255] range
# so the image is human-readable before we hand it off to the GPU for training.
_mean = np.array([123.675, 116.28, 103.53], dtype=np.float32)
_std  = np.array([ 58.395,  57.12,  57.375], dtype=np.float32)
pre_img_display  = np.clip(pre_img  * _std + _mean, 0, 255)
post_img_display = np.clip(post_img * _std + _mean, 0, 255)

# normalize=True so visualize_disaster_assessment divides by 255 → [0, 1] for imshow
visualize_disaster_assessment(pre_img_display, post_img_display, building_label, damage_label, normalize=True)

# Step 3: Create the Siamese U-Net model
Next we build our U-Net model with a siamese structure, loosely based on [Fully Convolutional Siamese Networks for Change Detection](https://arxiv.org/abs/1810.08462v1).

![Siamese-UNet-conc](https://i.postimg.cc/HnG8rR8K/2023-03-20-145159.png)

In [ ]:
# Building blocks of the U-Net architecture

# 1. Basic double-convolution block (used throughout the encoder and decoder)
class UNetConvolutionModule(nn.Module):
    def __init__(self, input_channels, output_channels):
        super().__init__()
        # bias=False: when Conv2d is immediately followed by BatchNorm, the bias is
        # mathematically redundant — BatchNorm's own learnable shift (β) absorbs it.
        # Removing it saves memory and avoids training a parameter that has no effect.
        self.conv1 = nn.Conv2d(input_channels, output_channels, kernel_size=3, padding=1, bias=False)
        self.bn1   = nn.BatchNorm2d(output_channels)
        self.relu1 = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv2d(output_channels, output_channels, kernel_size=3, padding=1, bias=False)
        self.bn2   = nn.BatchNorm2d(output_channels)
        self.relu2 = nn.ReLU(inplace=True)

    def forward(self, x):
        x = self.relu1(self.bn1(self.conv1(x)))
        x = self.relu2(self.bn2(self.conv2(x)))
        return x


# 2. Encoder step: halve spatial resolution, then apply double convolution
class UNetDownSamplingModule(nn.Module):
    def __init__(self, input_channels, output_channels):
        super().__init__()
        # MaxPool2d(2) halves H and W (stride=2 by default when kernel_size=2)
        self.down = nn.MaxPool2d(2)
        self.conv = UNetConvolutionModule(input_channels, output_channels)

    def forward(self, x):
        x = self.down(x)
        x = self.conv(x)
        return x


# 3. Decoder step: double spatial resolution, concatenate skip connection, then convolve
class UNetUpSamplingModule(nn.Module):
    def __init__(self, input_channels, output_channels):
        super().__init__()
        # Bilinear upsampling — smoother than transposed convolution and parameter-free
        self.up   = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
        self.conv = UNetConvolutionModule(input_channels, output_channels)

    def forward(self, x1, x2):
        # x1: upsampled feature map (from decoder)
        # x2: skip-connection feature map (from encoder, same resolution as upsampled x1)
        x1 = self.up(x1)

        # After upsampling, x1 may be 1 pixel smaller than x2 due to rounding in
        # odd-sized inputs. We pad x1 symmetrically so shapes match before concatenation.
        diffY = x2.size(2) - x1.size(2)
        diffX = x2.size(3) - x1.size(3)
        x1 = F.pad(x1, [diffX // 2, diffX - diffX // 2,
                         diffY // 2, diffY - diffY // 2])

        # Concatenate along the channel dimension (skip + decoder features)
        x = torch.cat([x2, x1], dim=1)
        return self.conv(x)


# 4. Final 1×1 convolution: map feature channels to the number of output classes
class UNetOutputConvolutionModule(nn.Module):
    def __init__(self, input_channels, output_channels):
        super().__init__()
        # kernel_size=1 acts as a per-pixel linear classifier across channels
        self.conv = nn.Conv2d(input_channels, output_channels, kernel_size=1)

    def forward(self, x):
        return self.conv(x)

The `UNet_conc` class is a **Siamese U-Net** that processes two images (pre- and post-disaster) and produces a per-pixel segmentation map. Here is how it works:

**Encoder (shared weights)**
Both images pass through the exact same encoder — the weights are literally shared, not just similar. This forces the network to learn a common feature space for both images, making it sensitive to *differences* rather than absolute appearance.

**Skip connections + feature concatenation**
At each encoder scale, the pre- and post-disaster feature maps are concatenated along the channel dimension before being passed to the decoder. This gives the decoder access to both "what it looked like before" and "what it looks like now" at every resolution level.

**Decoder**
A single decoder takes the concatenated features and progressively upsamples them back to the original image resolution, producing a segmentation mask.

**Why this design works for damage assessment**
- The shared encoder weights mean the network compares the two images in the same feature space, making change detection more reliable than using two separate encoders.
- Concatenation (rather than subtraction or averaging) preserves full information from both branches — the network learns which differences matter.
- Skip connections at multiple scales allow the model to reason about both fine-grained details (individual walls, windows) and coarse structure (whole buildings).

In [ ]:
class UNet_conc(nn.Module):
    """
    Siamese U-Net trained from scratch.
    Both branches share the same encoder weights (weight-sharing Siamese structure).
    Pre- and post-disaster features are concatenated at each scale in the decoder.
    """
    def __init__(self, input_channels, output_classes):
        super().__init__()
        self.n_channels = input_channels
        self.n_classes  = output_classes
        ch = [16, 32, 64, 128, 256]  # channel sizes at each encoder level

        # Encoder (shared between pre- and post-disaster branches)
        self.inc   = UNetConvolutionModule(input_channels, ch[0])
        self.down1 = UNetDownSamplingModule(ch[0], ch[1])
        self.down2 = UNetDownSamplingModule(ch[1], ch[2])
        self.down3 = UNetDownSamplingModule(ch[2], ch[3])
        self.down4 = UNetDownSamplingModule(ch[3], ch[4])

        # Decoder — input channels = concatenated pre+post features from both branches
        # up1: bottleneck (ch[4]*2) + skip (ch[3]*2) → ch[3]
        # up2: ch[3]     + skip (ch[2]*2) → ch[2]
        # up3: ch[2]     + skip (ch[1]*2) → ch[1]
        # up4: ch[1]     + skip (ch[0]*2) → ch[0]
        self.up1 = UNetUpSamplingModule(ch[4]*2 + ch[3]*2, ch[3])
        self.up2 = UNetUpSamplingModule(ch[3]   + ch[2]*2, ch[2])
        self.up3 = UNetUpSamplingModule(ch[2]   + ch[1]*2, ch[1])
        self.up4 = UNetUpSamplingModule(ch[1]   + ch[0]*2, ch[0])
        self.outc = UNetOutputConvolutionModule(ch[0], output_classes)

    def forward(self, pre, post):
        # --- Encode pre-disaster image ---
        pre1 = self.inc(pre)
        pre2 = self.down1(pre1)
        pre3 = self.down2(pre2)
        pre4 = self.down3(pre3)
        pre5 = self.down4(pre4)

        # --- Encode post-disaster image (same shared weights) ---
        post1 = self.inc(post)
        post2 = self.down1(post1)
        post3 = self.down2(post2)
        post4 = self.down3(post3)
        post5 = self.down4(post4)

        # --- Concatenate pre+post features at each scale along the channel dimension ---
        s1 = torch.cat((pre1, post1), dim=1)  # finest scale  (ch[0]*2 channels)
        s2 = torch.cat((pre2, post2), dim=1)
        s3 = torch.cat((pre3, post3), dim=1)
        s4 = torch.cat((pre4, post4), dim=1)
        s5 = torch.cat((pre5, post5), dim=1)  # coarsest scale (ch[4]*2 channels)

        # --- Decode: upsample + merge with skip connections ---
        x = self.up1(s5, s4)
        x = self.up2(x,  s3)
        x = self.up3(x,  s2)
        x = self.up4(x,  s1)

        return self.outc(x)

## Using a pretrained encoder

Training a U-Net from scratch requires a lot of labelled data and GPU time for the encoder to learn useful low-level features (edges, textures, shapes).

A better approach is **transfer learning**: replace the encoder with a ResNet-34 pretrained on ImageNet. The pretrained weights already encode rich visual features, so the network only needs to learn how to *use* those features for damage segmentation — not learn them from zero.

**What changes compared to `UNet_conc`:**
- The encoder is ResNet-34 (pretrained) instead of the custom down-sampling blocks.
- ResNet-34's first `conv1 + maxpool` downsamples by 4× before any residual stage, so two extra bilinear upsamplings are needed at the end of the decoder to recover the original resolution.
- Channel sizes follow ResNet-34's output dimensions: 64 → 128 → 256 → 512.
- Everything else (Siamese structure, skip connections, concatenation, decoder) remains identical.

In [ ]:
from torchvision import models
from torchvision.models import ResNet34_Weights

class UNet_conc_pretrained_resnet34(nn.Module):
    """
    Siamese U-Net with a pretrained ResNet-34 encoder.
    Using pretrained ImageNet weights gives the encoder a strong visual prior,
    which typically leads to faster convergence and better performance than
    training from scratch — especially when labelled data is limited.
    """
    def __init__(self, input_channels, output_classes, pretrained=True):
        super().__init__()
        self.n_channels = input_channels
        self.n_classes  = output_classes
        ch = [64, 128, 256, 512]  # ResNet-34 channel sizes per stage

        # Load ResNet-34 with pretrained ImageNet weights.
        # weights=None trains from scratch; ResNet34_Weights.IMAGENET1K_V1 uses pretrained.
        # Note: the old pretrained=True argument is deprecated in newer torchvision.
        weights = ResNet34_Weights.IMAGENET1K_V1 if pretrained else None
        resnet  = models.resnet34(weights=weights)

        # Reuse ResNet-34 layers as the shared encoder for both branches.
        # We split the network into stages to recover intermediate feature maps
        # needed for the U-Net skip connections.
        self.conv1   = nn.Sequential(resnet.conv1, resnet.bn1, resnet.relu)  # stride 2 → H/2
        self.maxpool = resnet.maxpool                                          # stride 2 → H/4
        self.layer1  = resnet.layer1   # 64  channels, H/4
        self.layer2  = resnet.layer2   # 128 channels, H/8
        self.layer3  = resnet.layer3   # 256 channels, H/16
        self.layer4  = resnet.layer4   # 512 channels, H/32

        # Bottleneck convolution applied after the deepest encoder features
        self.center = UNetConvolutionModule(ch[3], ch[3])

        # Decoder — same concatenation logic as UNet_conc but adjusted for ResNet channel sizes
        # up1: bottleneck (ch[3]*2) + skip (ch[3]*2) → ch[3]
        # up2: ch[3]     + skip (ch[2]*2) → ch[2]
        # up3: ch[2]     + skip (ch[1]*2) → ch[1]
        # up4: ch[1]     + skip (ch[0]*2) → ch[0]
        self.up1 = UNetUpSamplingModule(ch[3]*2 + ch[3]*2, ch[3])
        self.up2 = UNetUpSamplingModule(ch[3]   + ch[2]*2, ch[2])
        self.up3 = UNetUpSamplingModule(ch[2]   + ch[1]*2, ch[1])
        self.up4 = UNetUpSamplingModule(ch[1]   + ch[0]*2, ch[0])

        # Two bilinear upsamplings (×2 each = ×4 total) to recover the original resolution,
        # since ResNet's initial conv + maxpool downsampled by 4× before any residual stage.
        self.conv2 = nn.Sequential(
            nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True),
            UNetConvolutionModule(ch[0], ch[0] // 2),
            nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True),
        )
        self.outc = UNetOutputConvolutionModule(ch[0] // 2, output_classes)

    def forward(self, pre, post):
        # --- Encode pre-disaster image through ResNet-34 stages ---
        pre  = self.maxpool(self.conv1(pre))
        pre1 = self.layer1(pre)   # H/4,  64 ch
        pre2 = self.layer2(pre1)  # H/8,  128 ch
        pre3 = self.layer3(pre2)  # H/16, 256 ch
        pre4 = self.layer4(pre3)  # H/32, 512 ch
        pre5 = self.center(pre4)  # H/32, 512 ch (bottleneck)

        # --- Encode post-disaster image (same shared weights) ---
        post  = self.maxpool(self.conv1(post))
        post1 = self.layer1(post)
        post2 = self.layer2(post1)
        post3 = self.layer3(post2)
        post4 = self.layer4(post3)
        post5 = self.center(post4)

        # --- Concatenate pre+post features at each scale ---
        s1 = torch.cat((pre1, post1), dim=1)
        s2 = torch.cat((pre2, post2), dim=1)
        s3 = torch.cat((pre3, post3), dim=1)
        s4 = torch.cat((pre4, post4), dim=1)
        s5 = torch.cat((pre5, post5), dim=1)

        # --- Decode ---
        x = self.up1(s5, s4)
        x = self.up2(x,  s3)
        x = self.up3(x,  s2)
        x = self.up4(x,  s1)
        x = self.conv2(x)   # recover the ×4 lost in conv1 + maxpool

        return self.outc(x)


def get_model_UNET():
    # To train from scratch (no pretrained weights), use:
    # return UNet_conc(3, 2).to(device)
    # To use the pretrained ResNet-34 encoder (recommended):
    return UNet_conc_pretrained_resnet34(3, 2).to(device)

In [ ]:
# Free any GPU memory left over from previous runs before allocating the model.
# empty_cache() returns memory held by PyTorch's caching allocator back to the GPU.
# gc.collect() frees any Python objects on the CPU side that are no longer referenced.
# Together they give the model the maximum available memory on a fresh start.
torch.cuda.empty_cache()
gc.collect()

Now we have a UNet with a siamese structure, use the codes below to show it's weight and structure, and try to understand how the features in the model are transformed.

In [ ]:
model = get_model_UNET()  # already moves model to device internally

# Print the model architecture and parameter count.
# input_size matches the actual tile size used in training (256×256, 3 channels).
# Two inputs are passed because this is a Siamese network (pre + post disaster images).
# Look at: total params, output shape at each layer, and how channels evolve through
# the encoder (growing) and decoder (shrinking back to output_classes).
channels, height, width = 3, 256, 256
summary(model, input_size=[(channels, height, width), (channels, height, width)])

torch.cuda.empty_cache()
gc.collect()

# Step 4: Fit model

The xBD dataset has five damage categories:

| Class | Label | Meaning |
|-------|-------|---------|
| 0 | No building | Background pixels |
| 1 | No damage | Intact building |
| 2 | Minor damage | Slightly damaged |
| 3 | Major damage | Heavily damaged |
| 4 | Destroyed | Completely destroyed |

Training a 5-class model from scratch would require much more data and training time to converge well. Instead, we simplify the task to **binary classification**:

- **Class 0 — no damage**: merges "no building" (0) and "no damage" (1)
- **Class 1 — damaged**: merges "minor" (2), "major" (3), and "destroyed" (4)
- **Ignore index 255**: padding pixels from the crop augmentation are excluded from the loss

This binary formulation is still practically useful — it directly answers "is this building damaged?" — and converges much faster.

4. Set a Cross Entropy Loss and a [Adam Optmizer](https://pytorch.org/docs/stable/generated/torch.optim.Adam.html). Note that to get the model parameters you can use `model.parameters()`

In [ ]:
# F.cross_entropy is already available from the earlier import of torch.nn.functional as F

# Learning rate — 1e-3 is a standard starting point for Adam
init_learning_rate = 1e-3

# Adam combines momentum and adaptive learning rates, making it robust to
# different parameter scales and generally requiring less tuning than plain SGD
optimizer = torch.optim.Adam(model.parameters(), lr=init_learning_rate)

# LR scheduler: cosine annealing smoothly decays the learning rate from
# init_learning_rate down to ~0 over all epochs, helping the model settle
# into a sharp minimum rather than oscillating near convergence
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=20)

# Class weights for the loss — damaged pixels (class 1) are rare compared to
# background (class 0), so we upweight them to prevent the model from ignoring damage.
# [0.2, 1.0] means a damaged pixel contributes 5× more to the loss than a non-damaged one.
damage_class_weight = torch.FloatTensor([0.2, 1.]).to(device)

# Cross-entropy loss — standard choice for multi-class segmentation.
# ignore_index=255 will be used to exclude padded pixels from the loss computation.
criteria = F.cross_entropy

And, in order to be able to understand the predictive power of the model during training, we need to calculate the following accuracy metrics.
For details please look into the utils.metrics.

In [ ]:
import sys
# Add the path to the xBD utility library — this contains the Evaluator class
# used to compute damage F1 score during training and validation
sys.path.append("/kaggle/input/utilisxbd/utilsxbd")
from metrics import Evaluator
from tqdm import tqdm  # progress bar for the training loop

Set the number of times you want the model to see all of the training data:

In [ ]:
# Number of full passes through the training set.
# More epochs = more training, but risk of overfitting if validation F1 stops improving.
# Monitor the val F1 — if it plateaus or drops while train F1 keeps rising, stop early.
epochs = 20

And then lets start training!

In [ ]:
def merge_damage_classes(damage_label):
    """
    Remap 5-class xBD labels to binary in-place:
      0 (no building) → stays 0  ]  both become class 0 "no damage"
      1 (no damage)   → 0        ]
      2 (minor)       → 1        ]
      3 (major)       → 1        ]  all become class 1 "damaged"
      4 (destroyed)   → 1        ]
      255 (ignore)    → stays 255  excluded from the loss
    """
    damage_label[damage_label == 1] = 0
    damage_label[(damage_label > 1) & (damage_label < 255)] = 1
    return damage_label

In [ ]:
# Force NCCL initialisation before training starts.
# Without this, the very first model() call inside the loop blocks for ~60–120s
# while the two T4 GPUs negotiate, making the progress bar appear frozen.
print("Warming up DataParallel across GPUs...")
with torch.no_grad():
  dummy = torch.zeros(2, 3, 256, 256).to(device)
  _ = model(dummy, dummy)
  del dummy
torch.cuda.empty_cache()
print("Warmup done — starting training.")

In [ ]:
best_f1 = 0

# DataParallel splits each batch across all available GPUs automatically.
# If only one GPU is available it behaves like a normal model.
#model = torch.nn.DataParallel(model)

for epoch in range(epochs):
    print('Epoch: [{}/{}]'.format(epoch + 1, epochs))

    damage_evaluator_train = Evaluator(num_class=2, device=device)
    damage_evaluator_val   = Evaluator(num_class=2, device=device)

    # ------------------------------------------------------------------ #
    #  Training                                                            #
    # ------------------------------------------------------------------ #
    model.train()
    tbar = tqdm(train_loader)

    for data in tbar:
        pre_data, post_data, _, damage_label = data
        # Move tensors to GPU. collate_fn already returns long tensors for labels.
        pre_data     = pre_data.to(device)
        post_data    = post_data.to(device)
        damage_label = damage_label.to(device)

        # Remap 5 classes → 2 classes (binary damage detection)
        damage_label = merge_damage_classes(damage_label)

        # Forward pass
        predict_dam = model(pre_data, post_data)

        # Weighted cross-entropy — ignore padded pixels (255), upweight damaged class
        loss = criteria(predict_dam, damage_label, ignore_index=255, weight=damage_class_weight)

        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # Track F1 score on training set (accumulated over the epoch)
        predict_dam = torch.argmax(predict_dam, dim=1)
        damage_evaluator_train.add_batch(damage_label, predict_dam)
        tbar.set_description('Train — Loss: {:.4f} | F1: {:.4f}'.format(
            loss.item(), damage_evaluator_train.Damage_F1()))

    gc.collect()

    # ------------------------------------------------------------------ #
    #  Validation                                                          #
    # ------------------------------------------------------------------ #
    model.eval()
    tbar = tqdm(val_loader)

    with torch.no_grad():  # disable gradient computation — saves memory and speeds up inference
        for data in tbar:
            pre_data, post_data, _, damage_label = data
            pre_data     = pre_data.to(device)
            post_data    = post_data.to(device)
            damage_label = damage_label.to(device)

            damage_label = merge_damage_classes(damage_label)

            predict_dam = model(pre_data, post_data)
            loss = criteria(predict_dam, damage_label, ignore_index=255, weight=damage_class_weight)

            predict_dam = torch.argmax(predict_dam, dim=1)
            damage_evaluator_val.add_batch(damage_label, predict_dam)
            tbar.set_description('Val   — Loss: {:.4f} | F1: {:.4f}'.format(
                loss.item(), damage_evaluator_val.Damage_F1()))

    # Save the best model based on full-epoch validation F1 (not per-batch)
    epoch_f1 = damage_evaluator_val.Damage_F1()
    if epoch_f1 > best_f1:
        best_f1 = epoch_f1
        torch.save(model.state_dict(), 'best_model.pt')
        print('  ✓ New best model saved (val F1: {:.4f})'.format(best_f1))

    # Decay the learning rate according to the cosine schedule
    scheduler.step()

    torch.cuda.empty_cache()
    gc.collect()

# Save the final model regardless of whether it is the best
torch.save(model.state_dict(), 'final_model.pt')
print('Training complete. Best val F1: {:.4f}'.format(best_f1))